In [59]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter


In [60]:
df = kagglehub.load_dataset(
KaggleDatasetAdapter.PANDAS,
"mkechinov/ecommerce-behavior-data-from-multi-category-store",
"2019-Oct.csv",
pandas_kwargs={
"nrows": 10_000_000,
"dtype": {
'event_type': 'category',
'product_id': 'int32',
'category_code': 'category',
'brand': 'category',
'price': 'float32',
'user_id': 'int32',
}
}
)

/var/folders/z0/wpl2lwrs3rx3w51w3jxdmvfc0000gp/T/ipykernel_13128/1155051505.py:1: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000000 entries, 0 to 9999999
Data columns (total 9 columns):
 #   Column         Dtype   
---  ------         -----   
 0   event_time     object  
 1   event_type     category
 2   product_id     int32   
 3   category_id    int64   
 4   category_code  category
 5   brand          category
 6   price          float32 
 7   user_id        int32   
 8   user_session   object  
dtypes: category(3), float32(1), int32(2), int64(1), object(2)
memory usage: 381.6+ MB


In [62]:
df

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.790001,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.200001,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.099976,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.740005,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.979980,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d
...,...,...,...,...,...,...,...,...,...
9999995,2019-10-08 17:26:09 UTC,view,22900082,2053013561780732677,furniture.bedroom.pillow,belashoff,25.740000,518471106,708497d0-bed7-476b-af47-66ebda1d6c68
9999996,2019-10-08 17:26:09 UTC,view,32100001,2060237588744111062,NaN,NaN,164.100006,514437524,f8fc5c52-ae97-4a64-b6f4-fa3f5f9f28df
9999997,2019-10-08 17:26:09 UTC,view,26400265,2053013563651392361,NaN,lucente,154.190002,547749418,e22d0270-8dea-4cc1-ae99-e78a581f515e
9999998,2019-10-08 17:26:09 UTC,view,1005125,2053013555631882655,electronics.smartphone,apple,1955.010010,555334348,28b16eb3-1362-4806-8f4f-0265c7afe096


In [63]:
df['event_type'].value_counts(normalize=True)

event_type
view        0.962782
cart        0.019935
purchase    0.017283
Name: proportion, dtype: float64

This specific retailer doesn't use remove_from_cart event. They are using override cart mechanics in API. So you can figured out removed from cart products as difference between purchase and cart events.

In [64]:
df['category_id'].nunique()

566

In [65]:
df['product_id'].nunique()

121903

In [66]:
df['category_code'].nunique()

123

In [67]:
# Stats descriptives (colonnes numériques)
df.describe()

,product_id,category_id,price,user_id
count,1.000000e+07,1.000000e+07,1.000000e+07,1.000000e+07
mean,1.011534e+07,2.056684e+18,2.960614e+02,5.317569e+08
std,1.124471e+07,1.681746e+16,3.669100e+02,1.715918e+07
min,1.001588e+06,2.053014e+18,0.000000e+00,1.835035e+08
25%,1.005115e+06,2.053014e+18,6.525000e+01,5.156193e+08
50%,4.900087e+06,2.053014e+18,1.619100e+02,5.274672e+08
75%,1.570007e+07,2.053014e+18,3.618200e+02,5.487683e+08
max,5.560003e+07,2.175420e+18,2.574070e+03,5.581647e+08


In [68]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'missing': missing, '%': missing_pct}).query('missing > 0').sort_values('%', ascending=False)

,missing,%
category_code,3224119,32.2
brand,1388005,13.9
user_session,1,0.0


In [69]:
df['user_id'].nunique()

1070614

In [70]:
df_purchase = df.groupby('category_code')['event_type'].apply(lambda x: (x == 'purchase').sum())

/var/folders/z0/wpl2lwrs3rx3w51w3jxdmvfc0000gp/T/ipykernel_13128/985662042.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_purchase = df.groupby('category_code')['event_type'].apply(lambda x: (x == 'purchase').sum())


In [71]:
df_purchase

category_code
accessories.bag         265
accessories.umbrella      8
accessories.wallet       54
apparel.belt              2
apparel.costume          25
                       ... 
sport.ski                 2
sport.snowboard           3
sport.tennis              0
sport.trainer            65
stationery.cartrige      19
Name: event_type, Length: 123, dtype: int64

In [72]:
df_purchase.sort_values(ascending=False)

category_code
electronics.smartphone           79205
electronics.audio.headphone       7257
electronics.video.tv              4849
electronics.clocks                4495
computers.notebook                3914
                                 ...  
country_yard.furniture.bench         0
country_yard.furniture.hammok        0
apparel.dress                        0
apparel.jacket                       0
sport.tennis                         0
Name: event_type, Length: 123, dtype: int64

In [73]:
(df_purchase / df_purchase.sum() * 100).sort_values(ascending=False)

category_code
electronics.smartphone           59.461428
electronics.audio.headphone       5.448035
electronics.video.tv              3.640281
electronics.clocks                3.374523
computers.notebook                2.938350
                                   ...    
country_yard.furniture.bench      0.000000
country_yard.furniture.hammok     0.000000
apparel.dress                     0.000000
apparel.jacket                    0.000000
sport.tennis                      0.000000
Name: event_type, Length: 123, dtype: float64

In [74]:
# Lister les catégories à 0 achats
df_purchase[df_purchase == 0].index.tolist()
# Combien il y en a
(df_purchase == 0).sum()

np.int64(7)

In [75]:
# Catégories avec au moins 1 achat
df_purchase_clean = df_purchase[df_purchase > 0].sort_values(ascending=False)
df_purchase_clean

category_code
electronics.smartphone         79205
electronics.audio.headphone     7257
electronics.video.tv            4849
electronics.clocks              4495
computers.notebook              3914
                               ...  
apparel.scarf                      2
apparel.shoes.sandals              2
apparel.shorts                     2
apparel.shoes.ballet_shoes         1
apparel.skirt                      1
Name: event_type, Length: 116, dtype: int64

# Preprocessing

### Stratégie drop catégorie code & Brand

In [76]:

X = df.drop(columns=['category_code', 'brand'])

In [77]:
X

,event_time,event_type,product_id,category_id,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,35.790001,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,33.200001,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,543.099976,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,251.740005,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,1081.979980,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d
...,...,...,...,...,...,...,...
9999995,2019-10-08 17:26:09 UTC,view,22900082,2053013561780732677,25.740000,518471106,708497d0-bed7-476b-af47-66ebda1d6c68
9999996,2019-10-08 17:26:09 UTC,view,32100001,2060237588744111062,164.100006,514437524,f8fc5c52-ae97-4a64-b6f4-fa3f5f9f28df
9999997,2019-10-08 17:26:09 UTC,view,26400265,2053013563651392361,154.190002,547749418,e22d0270-8dea-4cc1-ae99-e78a581f515e
9999998,2019-10-08 17:26:09 UTC,view,1005125,2053013555631882655,1955.010010,555334348,28b16eb3-1362-4806-8f4f-0265c7afe096


### Changement de stratégie, on garde catégorie code mais on filtre par top 5 de catégories code (électronique)

In [78]:
top5 = ['electronics.smartphone', 'electronics.audio.headphone', 'electronics.video.tv',
        'electronics.clocks', 'computers.notebook']

X = df[df['category_code'].isin(top5)]

In [79]:
X

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.740005,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.979980,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d
9,2019-10-01 00:00:11 UTC,view,1004545,2053013555631882655,electronics.smartphone,huawei,566.010010,537918940,406c46ed-90a4-4787-a43b-59a410c1a5fb
11,2019-10-01 00:00:11 UTC,view,1005011,2053013555631882655,electronics.smartphone,samsung,900.640015,530282093,50a293fb-5940-41b2-baf3-17af0e812101
16,2019-10-01 00:00:18 UTC,view,1801995,2053013554415534427,electronics.video.tv,haier,193.029999,537192226,e3151795-c355-4efa-acf6-e1fe1bebeee5
...,...,...,...,...,...,...,...,...,...
9999984,2019-10-08 17:26:08 UTC,view,1005105,2053013555631882655,electronics.smartphone,apple,1415.479980,512565865,4f98f4f3-60da-48aa-bc71-0d39b2eba526
9999985,2019-10-08 17:26:08 UTC,view,1307383,2053013558920217191,computers.notebook,asus,1282.140015,513451334,c2f51343-7977-4ac3-ae5a-5e274918310e
9999987,2019-10-08 17:26:08 UTC,view,1307293,2053013558920217191,computers.notebook,msi,866.710022,513956748,fdf662f6-e9a4-4991-9d2e-a8b5b6073441
9999989,2019-10-08 17:26:08 UTC,view,1004433,2053013555631882655,electronics.smartphone,samsung,257.149994,512963963,b61ab7c6-2a1a-4ef2-9e1d-1fa9b288fe9b


In [83]:
X = X.drop_duplicates()

In [ ]:
# Il reste 38% des lignes, donc 1/3 des events sont compris sur ces 5 catégories

In [84]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3857591 entries, 3 to 9999998
Data columns (total 9 columns):
 #   Column         Dtype   
---  ------         -----   
 0   event_time     object  
 1   event_type     category
 2   product_id     int32   
 3   category_id    int64   
 4   category_code  category
 5   brand          category
 6   price          float32 
 7   user_id        int32   
 8   user_session   object  
dtypes: category(3), float32(1), int32(2), int64(1), object(2)
memory usage: 176.7+ MB


In [85]:
missing = X.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'missing': missing, '%': missing_pct}).query('missing > 0').sort_values('%', ascending=False)

,missing,%
brand,65254,0.7
user_session,1,0.0
